<a href="https://colab.research.google.com/github/Fugant1/OPERA/blob/main/ser_grpo_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cloud Connection
Drive

In [ ]:
from pathlib import Path

In [ ]:
def mount_drive():
    """Mount Google Drive in Colab."""
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
mount_drive()

Mounted at /content/drive


# Get Audio Data
MELD

## Raw

In [ ]:
from pathlib import Path
import os
import subprocess
import time
import pandas as pd

In [ ]:

DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SER_DPO/MELD/MELD.tar")
MELD_DIR = Path("/content/MELD")
TRAIN_DIR = MELD_DIR / "preprocessed_train_splits"

In [ ]:
def extract_tar(archive_path: Path, destination: Path):
    """Extract a .tar archive to destination."""
    archive_path = Path(archive_path)
    destination = Path(destination)

    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    destination.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {archive_path.name} → {destination}")
    start = time.time()

    subprocess.run(
        ["tar", "-xf", str(archive_path), "-C", str(destination)],
        check=True,
    )

    elapsed = time.time() - start
    print(f"Extraction completed in {elapsed:.1f}s")

In [ ]:

def dataset_info(path: Path):
    """Print basic information about an extracted dataset."""
    path = Path(path)

    n_files = sum(1 for p in path.rglob("*") if p.is_file())

    size = subprocess.run(
        ["du", "-sh", str(path)],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()[0]

    print(f"Files: {n_files:,}")
    print(f"Size:  {size}")

In [ ]:
def find_file(root: Path, pattern: str) -> Path:
    """Find the first file matching a pattern."""
    matches = list(Path(root).rglob(pattern))

    if not matches:
        raise FileNotFoundError(
            f"Could not find '{pattern}' under {root}"
        )

    return matches[0]


In [ ]:
def load_meld_split(split_dir: Path, split: str = "train") -> pd.DataFrame:
    """
    Load a MELD split and resolve audio paths.

    Expected files:
        {split}_preprocessed.csv
        {split}_sent_emo.csv
        *.wav
    """
    split_dir = Path(split_dir)

    audio_files = list(split_dir.rglob("*.wav"))

    preprocessed_csv = find_file(
        split_dir,
        f"{split}_preprocessed.csv",
    )

    emotion_csv = find_file(
        split_dir,
        f"{split}_sent_emo.csv",
    )

    df = pd.read_csv(preprocessed_csv)
    df_emo = pd.read_csv(emotion_csv)

    # Resolve audio filenames → absolute paths
    audio_map = {
        path.name: str(path.resolve())
        for path in audio_files
    }

    df["audio_path"] = (
        df["audio_path"]
        .astype(str)
        .map(lambda x: Path(x).name)
        .map(audio_map)
    )

    # Normalize merge keys
    df["dialogue"] = df["dialogue"].astype(int)
    df["utterance"] = df["utterance"].astype(int)

    df_emo["Dialogue_ID"] = df_emo["Dialogue_ID"].astype(int)
    df_emo["Utterance_ID"] = df_emo["Utterance_ID"].astype(int)

    # Add MELD metadata
    df = df.merge(
        df_emo[
            [
                "Dialogue_ID",
                "Utterance_ID",
                "Emotion",
                "Utterance",
                "Speaker",
            ]
        ],
        left_on=["dialogue", "utterance"],
        right_on=["Dialogue_ID", "Utterance_ID"],
        how="left",
    )

    df = (
        df.drop(columns=["Dialogue_ID", "Utterance_ID"])
          .rename(
              columns={
                  "Utterance": "text",
                  "Emotion": "emotion",
                  "Speaker": "speaker",
              }
          )
    )

    # Validation
    missing_audio = df["audio_path"].isna().sum()
    missing_emotion = df["emotion"].isna().sum()

    print(f"Loaded {len(df):,} samples")
    print(
        f"Audio matched: "
        f"{len(df) - missing_audio:,}/{len(df):,}"
    )

    if missing_audio:
        print(f"Warning: {missing_audio:,} audio files missing")

    print(
        f"Metadata matched: "
        f"{len(df) - missing_emotion:,}/{len(df):,}"
    )

    if missing_emotion:
        print(f"Warning: {missing_emotion:,} metadata rows missing")

    return df

In [ ]:
def load_meld(
    archive_path: Path = DRIVE_ARCHIVE,
    destination: Path = MELD_DIR,
    split: str = "train",
) -> pd.DataFrame:
    """Download/extract and load a MELD split."""

    if not destination.exists():
        extract_tar(archive_path, destination)
    else:
        print(f"Dataset already exists: {destination}")

    dataset_info(destination)

    split_dir = destination / f"preprocessed_{split}_splits"

    if not split_dir.exists():
        raise FileNotFoundError(
            f"Split directory not found: {split_dir}"
        )

    return load_meld_split(split_dir, split)

In [ ]:
raw_train_set = load_meld(
    archive_path=DRIVE_ARCHIVE,
    destination=MELD_DIR,
    split="train",
)
train_set = raw_train_set

In [ ]:
display(raw_train_set.head())

In [ ]:
raw_train_set['emotion'].value_counts()

In [ ]:
raw_train_set['speaker'].value_counts()

## Acoustic Cues Analysis (Raw Train Set)

In [ ]:
# ==============================================================================
# Extração / Carregamento dos Cues Acústicos (eGeMAPSv02) no Raw Train Set
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")

import sys
import subprocess
from pathlib import Path
from typing import Dict, List, Any
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Mapeamento dos descritores do Acoustic Inventory para variáveis físicas do eGeMAPS
ACOUSTIC_CUES = {
    "Pitch Height (st)": "F0semitoneFrom27.5Hz_sma3nz_amean",
    "Pitch Dynamics (st)": "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "Vocal Energy (Sones)": "loudness_sma3_percentile50.0",
    "Speaking Rate (seg/s)": "VoicedSegmentsPerSec",
    "Spectral Tilt (dB)": "hammarbergIndexV_sma3nz_amean",
    "HNR (dB)": "HNRdBACF_sma3nz_amean",
    "Jitter Local": "jitterLocal_sma3nz_amean",
    "Shimmer Local (dB)": "shimmerLocaldB_sma3nz_amean",
}

target_features = list(ACOUSTIC_CUES.values())
missing_features = [col for col in target_features if col not in raw_train_set.columns]

if missing_features:
    cache_path = Path("/content/train_egemaps_features.csv")
    alt_cache_path = Path("/content/raw_train_egemaps_features.csv")
    active_cache = cache_path if cache_path.exists() else (alt_cache_path if alt_cache_path.exists() else None)

    if active_cache is not None:
        print(f"📁 Carregando features eGeMAPSv02 cacheadas de {active_cache}...")
        df_cached = pd.read_csv(active_cache)
        if "audio_path" in df_cached.columns:
            cols_to_merge = ["audio_path"] + [c for c in target_features if c in df_cached.columns and c not in raw_train_set.columns]
            raw_train_set = raw_train_set.merge(df_cached[cols_to_merge].drop_duplicates(subset=["audio_path"]), on="audio_path", how="left")
        else:
            cols_to_add = [c for c in missing_features if c in df_cached.columns]
            raw_train_set = pd.concat([raw_train_set.reset_index(drop=True), df_cached[cols_to_add].reset_index(drop=True)], axis=1)
    else:
        print("🎙️ Extraindo features acústicas eGeMAPSv02 via openSMILE no raw train set...")
        try:
            import opensmile
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opensmile"], check=True)
            import opensmile

        smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )

        extracted_records = []
        valid_indices = []
        for idx, row in tqdm(raw_train_set.iterrows(), total=len(raw_train_set), desc="Processando Áudios (Raw Train)"):
            p = Path(str(row["audio_path"]))
            if p.is_file():
                try:
                    feat = smile.process_file(str(p))
                    extracted_records.append(feat.iloc[0].to_dict())
                    valid_indices.append(idx)
                except Exception:
                    pass

        if extracted_records:
            df_feats = pd.DataFrame(extracted_records, index=valid_indices)
            for c in target_features:
                if c in df_feats.columns:
                    raw_train_set.loc[valid_indices, c] = df_feats[c]

            try:
                df_feats["audio_path"] = raw_train_set.loc[valid_indices, "audio_path"].values
                df_feats.to_csv(cache_path, index=False)
                df_feats.to_csv(alt_cache_path, index=False)
                print(f"✅ Cache salvo em {cache_path}")
            except Exception as e:
                print(f"Aviso ao salvar cache: {e}")
        else:
            print("⚠️ Arquivos de áudio não encontrados para extração imediata. Verifique se o MELD.tar foi extraído.")

train_set = raw_train_set

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy

def analyze_speaker_quantile_validity(df: pd.DataFrame, min_utterances_threshold: int = 8):
    # Detecta coluna de emoção
    label_col = "emotion" if "emotion" in df.columns else ("label" if "label" in df.columns else None)
    if label_col is None:
        raise ValueError("Coluna de emoção ('emotion' ou 'label') não encontrada no DataFrame.")

    total_samples = len(df)
    spk_counts = df['speaker'].value_counts()
    num_unique_speakers = len(spk_counts)

    print("=" * 80)
    print(f"📊 DIAGNÓSTICO DE VALIDADE DOS QUANTIS POR LOCUTOR (Total: {total_samples} amostras)")
    print("=" * 80)

    # ---------------------------------------------------------
    # 1. ANÁLISE DE VOLUME / CAUDA LONGA (Utterances por Speaker)
    # ---------------------------------------------------------
    bins = [0, 1, 3, 7, 20, np.inf]
    bin_labels = [
        "1 fala (100% degenerado)",
        "2 a 3 falas (Crítico)",
        f"4 a {min_utterances_threshold - 1} falas (Abaixo do threshold)",
        f"{min_utterances_threshold} a 20 falas (Limítrofe)",
        "> 20 falas (Estatisticamente estável)"
    ]

    binned_speakers = pd.cut(spk_counts, bins=bins, labels=bin_labels)
    spk_dist = pd.DataFrame({
        "Qtd Falantes": binned_speakers.value_counts(sort=False),
        "% dos Falantes": (binned_speakers.value_counts(sort=False, normalize=True) * 100).round(2),
    })

    # Calcula quantas falas totais do dataset caem em cada balde
    utterance_sums = []
    for interval_label in bin_labels:
        matching_spks = spk_counts[binned_speakers == interval_label].index
        utterance_sums.append(df[df['speaker'].isin(matching_spks)].shape[0])

    spk_dist["Falas Afetadas"] = utterance_sums
    spk_dist["% do Corpus Total"] = ((spk_dist["Falas Afetadas"] / total_samples) * 100).round(2)

    print("\n1️⃣ DISTRIBUIÇÃO DE FALAS E COBERTURA DE LOCUTORES:")
    display(spk_dist)

    speakers_below_threshold = (spk_counts < min_utterances_threshold).sum()
    pct_fallback = (df['speaker'].isin(spk_counts[spk_counts < min_utterances_threshold].index).sum() / total_samples) * 100

    print(f"• Total de locutores únicos: {num_unique_speakers}")
    print(f"• Locutores com menos de {min_utterances_threshold} falas: {speakers_below_threshold} ({speakers_below_threshold / num_unique_speakers * 100:.1f}%)")
    print(f"• Impacto no dataset: {pct_fallback:.2f}% de todas as falas caem no Fallback Global (__GLOBAL__).")

    # ---------------------------------------------------------
    # 2. ANÁLISE DE DIVERSIDADE EMOCIONAL (Viés de Amostragem)
    # ---------------------------------------------------------
    # Para falantes com >= min_utterances_threshold, calcula se o perfil está enviesado
    eligible_speakers = spk_counts[spk_counts >= min_utterances_threshold].index
    num_classes = df[label_col].nunique()
    max_entropy = np.log2(num_classes)  # Entropia máxima teórica (uniforme)

    diversity_records = []
    for spk in spk_counts.index:
        sub = df[df['speaker'] == spk]
        counts = sub[label_col].value_counts()
        probs = counts / len(sub)

        # Entropia normalizada [0.0 = mono-emoção, 1.0 = perfeitamente balanceado]
        shannon_ent = entropy(probs, base=2) / max_entropy if max_entropy > 0 else 0.0
        dominant_emo = counts.index[0]
        dominant_pct = (counts.iloc[0] / len(sub)) * 100

        diversity_records.append({
            "speaker": spk,
            "total_utterances": len(sub),
            "dominant_emotion": dominant_emo,
            "dominant_pct": round(dominant_pct, 1),
            "normalized_entropy": round(shannon_ent, 3),
            "status_calibracao": "Individual" if len(sub) >= min_utterances_threshold else "Fallback Global"
        })

    df_div = pd.DataFrame(diversity_records)

    # Locutores 'elegíveis' mas com forte skew emocional (ex: > 60% uma única emoção)
    skewed_calibrated = df_div[(df_div["status_calibracao"] == "Individual") & (df_div["dominant_pct"] >= 60.0)]

    print("\n" + "-" * 80)
    print("2️⃣ ANÁLISE DE VIÉS EMOCIONAL NOS LOCUTORES CALIBRADOS:")
    print(f"• Locutores calibrados individualmente: {len(eligible_speakers)}")
    print(f"• Locutores calibrados com SEVERO VIÉS emocional (>= 60% em uma única emoção): {len(skewed_calibrated)}")

    if len(skewed_calibrated) > 0:
        print("\n⚠️ ALERTA: Locutores calibrados com viés (seus quantis refletem a emoção dominante, não a voz neutra):")
        display(skewed_calibrated.sort_values(by="total_utterances", ascending=False).head(10))

    # ---------------------------------------------------------
    # 3. VERIFICAÇÃO DE COLAPSO DA FAIXA MODERADA (Se houver eGeMAPS)
    # ---------------------------------------------------------
    if "F0semitoneFrom27.5Hz_sma3nz_amean" in df.columns:
        print("\n" + "-" * 80)
        print("3️⃣ VERIFICAÇÃO DE COLAPSO DE QUANTIS (Largura da Faixa Moderada em Pitch):")
        f0_col = "F0semitoneFrom27.5Hz_sma3nz_amean"

        iqr_records = []
        for spk in eligible_speakers[:15]:  # Primeiros 15
            sub_f0 = df[df['speaker'] == spk][f0_col]
            q33 = sub_f0.quantile(0.33)
            q66 = sub_f0.quantile(0.66)
            band_width = q66 - q33  # Largura da faixa considerada 'moderate'

            iqr_records.append({
                "speaker": spk,
                "n_utterances": len(sub_f0),
                "Q33_semitones": round(q33, 2),
                "Q66_semitones": round(q66, 2),
                "Bandwidth_Moderate": round(band_width, 2),
                "Bandwidth_Colapsada": "SIM (Muito Estreita)" if band_width < 1.0 else "NÃO (Saudável)"
            })
        display(pd.DataFrame(iqr_records))

    print("=" * 80)
    return df_div

# Executa o diagnóstico sobre o raw_train_set
df_speaker_analysis = analyze_speaker_quantile_validity(raw_train_set, min_utterances_threshold=8)

In [ ]:
# ==============================================================================
# Diagnóstico de Variação por Emoção e Amplitudes dos Locutores (Raw Train Set)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_emotion_and_speaker_acoustics(
    df: pd.DataFrame,
    top_n_speakers: int = 20,
    cues_dict: Dict[str, str] = ACOUSTIC_CUES,
):
    active_cues = {label: col for label, col in cues_dict.items() if col in df.columns}
    if not active_cues:
        print("⚠️ Nenhuma das colunas de cues acústicos foi encontrada no DataFrame.")
        return

    print("=" * 90)
    print("📊 DIAGNÓSTICO ACÚSTICO: VARIAÇÃO POR EMOÇÃO, QUANTIS E AMPLITUDES (LOCUTORES VS GLOBAL)")
    print("=" * 90)

    # A. AMPLITUDE E QUANTIS GLOBAIS (Full Corpus)
    global_rows = []
    global_amps = {}
    for label, col in active_cues.items():
        s = df[col].dropna()
        q10, q25, q33, q50, q66, q75, q90 = s.quantile([0.10, 0.25, 0.33, 0.50, 0.66, 0.75, 0.90])
        min_v, max_v = s.min(), s.max()
        amp = max_v - min_v
        global_amps[label] = amp
        global_rows.append({
            "Metric": label,
            "Count": len(s),
            "Mean": round(s.mean(), 2),
            "Std": round(s.std(), 2),
            "Min": round(min_v, 2),
            "Q10": round(q10, 2),
            "Q25": round(q25, 2),
            "Q33": round(q33, 2),
            "Median (Q50)": round(q50, 2),
            "Q66": round(q66, 2),
            "Q75": round(q75, 2),
            "Q90": round(q90, 2),
            "Max": round(max_v, 2),
            "Global Amplitude": round(amp, 2),
            "Global IQR": round(q75 - q25, 2),
        })

    df_global = pd.DataFrame(global_rows).set_index("Metric")

    # B. VARIAÇÃO DE EMOÇÃO E NÚMEROS DOS CUES ACÚSTICOS EM QUANTIS
    emo_rows = []
    for emo, emo_df in df.groupby("emotion"):
        for label, col in active_cues.items():
            s = emo_df[col].dropna()
            if len(s) == 0:
                continue
            q10, q25, q33, q50, q66, q75, q90 = s.quantile([0.10, 0.25, 0.33, 0.50, 0.66, 0.75, 0.90])
            min_v, max_v = s.min(), s.max()
            emo_rows.append({
                "Emotion": emo,
                "Metric": label,
                "Utterances": len(s),
                "Mean": round(s.mean(), 2),
                "Std": round(s.std(), 2),
                "Min": round(min_v, 2),
                "Q25": round(q25, 2),
                "Q33": round(q33, 2),
                "Median (Q50)": round(q50, 2),
                "Q66": round(q66, 2),
                "Q75": round(q75, 2),
                "Max": round(max_v, 2),
                "Amplitude": round(max_v - min_v, 2),
                "IQR": round(q75 - q25, 2),
            })

    df_emotion_quantiles = pd.DataFrame(emo_rows)

    # C. AMPLITUDE DOS 20 LOCUTORES COM MAIS FALAS (TOP 20 UTTERANCE SPEAKERS)
    spk_counts = df["speaker"].value_counts()
    top_speakers = spk_counts.head(top_n_speakers).index.tolist()

    spk_amp_rows = []
    spk_ratio_rows = []

    for spk in top_speakers:
        sub = df[df["speaker"] == spk]
        n_utt = len(sub)
        amp_entry = {"Speaker": spk, "Utterances": n_utt}
        ratio_entry = {"Speaker": f"{spk} (n={n_utt})"}

        for label, col in active_cues.items():
            s = sub[col].dropna()
            g_amp = global_amps.get(label, 0)
            if len(s) > 1:
                s_amp = s.max() - s.min()
                amp_entry[label] = round(s_amp, 2)
                ratio_entry[label] = round((s_amp / g_amp) * 100, 1) if g_amp > 0 else 0.0
            else:
                amp_entry[label] = 0.0
                ratio_entry[label] = 0.0

        spk_amp_rows.append(amp_entry)
        spk_ratio_rows.append(ratio_entry)

    df_top20_amplitudes = pd.DataFrame(spk_amp_rows).set_index("Speaker")
    df_top20_relative_amplitudes = pd.DataFrame(spk_ratio_rows).set_index("Speaker")

    # Adiciona estatísticas agregadas dos locutores à tabela global
    metric_cols = [c for c in df_top20_amplitudes.columns if c != "Utterances"]
    df_global["Mean Speaker Amp"] = [
        round(df_top20_amplitudes[m].mean(), 2) for m in df_global.index if m in metric_cols
    ]
    df_global["Speaker/Global Amp (%)"] = [
        round((df_top20_amplitudes[m].mean() / df_global.loc[m, "Global Amplitude"]) * 100, 1)
        for m in df_global.index if m in metric_cols
    ]

    # D. EXIBIÇÃO DE RESULTADOS
    print("\n1️⃣ QUANTIS E AMPLITUDE GLOBAL POR CUE ACÚSTICO (CORPUS COMPLETO):")
    display(df_global[["Min", "Q25", "Q33", "Median (Q50)", "Q66", "Q75", "Max", "Global Amplitude", "Global IQR", "Mean Speaker Amp", "Speaker/Global Amp (%)"]])

    print("\n2️⃣ VARIAÇÃO DE EMOÇÕES NOS QUANTIS E AMPLITUDES DOS CUES ACÚSTICOS:")
    display(df_emotion_quantiles.head(16))

    print(f"\n3️⃣ AMPLITUDES ABSOLUTAS (MAX - MIN) DOS {top_n_speakers} LOCUTORES COM MAIS FALAS:")
    display(df_top20_amplitudes)

    print(f"\n4️⃣ COBERTURA DA AMPLITUDE GLOBAL (%) PELOS {top_n_speakers} LOCUTORES COM MAIS FALAS:")
    display(df_top20_relative_amplitudes)

    # --------------------------------------------------------------------------
    # 4. PLOTS
    # --------------------------------------------------------------------------
    print("\n" + "=" * 90)
    print("📈 GERANDO VISUALIZAÇÕES DOS QUANTIS E AMPLITUDES...")
    print("=" * 90)

    # FIGURA 1: Variação das Emoções por Cue Acústico (Boxplots com Quantis)
    n_cues = len(active_cues)
    cols = 2
    rows = (n_cues + cols - 1) // cols

    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    fig1, axes1 = plt.subplots(rows, cols, figsize=(16, 4.2 * rows))
    axes1 = np.array(axes1).flatten()
    emo_order = sorted(df["emotion"].unique())

    for idx, (label, col) in enumerate(active_cues.items()):
        ax = axes1[idx]
        sns.boxplot(
            data=df,
            x="emotion",
            y=col,
            order=emo_order,
            hue="emotion",
            legend=False,
            palette="Set2",
            ax=ax,
            fliersize=2,
            linewidth=1.2,
        )
        ax.set_title(f"Acoustic Variation Across Emotions: {label}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Emotion", fontsize=9)
        ax.set_ylabel(label, fontsize=9)
        ax.tick_params(axis="x", rotation=20)

    for idx in range(len(active_cues), len(axes1)):
        fig1.delaxes(axes1[idx])

    fig1.suptitle("Distributions and Quantiles of Acoustic Cues Across Emotions", fontsize=14, fontweight="bold", y=1.002)
    plt.tight_layout()
    plt.show()

    # FIGURA 2: Comparação de Amplitude dos Locutores vs Amplitude Global
    fig2, (ax_heat, ax_dist) = plt.subplots(1, 2, figsize=(22, 10), gridspec_kw={"width_ratios": [1.3, 1]})

    # Subplot A: Heatmap com percentual de amplitude global coberto
    sns.heatmap(
        df_top20_relative_amplitudes,
        annot=True,
        fmt=".1f",
        cmap="YlGnBu",
        cbar_kws={"label": "Speaker Amplitude / Global Amplitude (%)"},
        vmin=20,
        vmax=100,
        ax=ax_heat,
        linewidths=0.5,
    )
    ax_heat.set_title(f"Top {top_n_speakers} Speakers: Dynamic Range Coverage (% of Global Amplitude)", fontsize=13, fontweight="bold")
    ax_heat.set_ylabel("Speaker (Utterances)", fontsize=11)
    ax_heat.tick_params(axis="x", rotation=25)

    # Subplot B: Distribuição da cobertura relativa por métrica acústica
    df_melted_ratio = df_top20_relative_amplitudes.melt(var_name="Metric", value_name="Relative Amplitude (%)")
    sns.boxplot(
        data=df_melted_ratio,
        x="Metric",
        y="Relative Amplitude (%)",
        hue="Metric",
        legend=False,
        palette="mako",
        ax=ax_dist,
        linewidth=1.2,
    )
    sns.stripplot(
        data=df_melted_ratio,
        x="Metric",
        y="Relative Amplitude (%)",
        color="crimson",
        alpha=0.6,
        jitter=0.2,
        size=5,
        ax=ax_dist,
    )
    ax_dist.axhline(100.0, color="red", linestyle="--", linewidth=1.8, label="Global Amplitude (100%)")
    ax_dist.set_ylim(0, 110)
    ax_dist.set_title("Distribution of Speaker Coverage (%) Across Acoustic Metrics", fontsize=13, fontweight="bold")
    ax_dist.set_xlabel("Acoustic Metric", fontsize=11)
    ax_dist.set_ylabel("Speaker Amplitude / Global Amplitude (%)", fontsize=11)
    ax_dist.tick_params(axis="x", rotation=25)
    ax_dist.legend(loc="lower right", fontsize=10)

    fig2.suptitle(f"Amplitude Analysis: Top {top_n_speakers} Most Frequent Speakers vs. Global Amplitude", fontsize=14, fontweight="bold", y=0.995)
    plt.tight_layout()
    plt.show()

    return {
        "global_summary": df_global,
        "emotion_quantiles": df_emotion_quantiles,
        "speaker_amplitudes": df_top20_amplitudes,
        "speaker_relative_amplitudes": df_top20_relative_amplitudes,
    }

# Executa o diagnóstico completo sobre raw_train_set
acoustic_analysis_results = analyze_emotion_and_speaker_acoustics(raw_train_set, top_n_speakers=20)

## Balanced

In [ ]:
import pandas as pd
from pathlib import Path
import librosa


def get_audio_duration(audio_path: str) -> float | None:
    """Return audio duration in seconds, or None if the file cannot be read."""
    try:
        return librosa.get_duration(path=audio_path)
    except Exception:
        return None


def sample_balanced_meld(
    df: pd.DataFrame,
    target_total: int = 2000,
    max_neutral_pct: float = 0.15,
    min_duration: float = 0.5,
    max_duration: float = 30.0,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Create a deterministic, emotion-balanced MELD subset.

    Filtering:
        - Removes missing/invalid audio files.
        - Removes audio shorter than min_duration.
        - Removes audio longer than max_duration.

    Balancing:
        - Caps neutral at max_neutral_pct.
        - Distributes remaining samples across non-neutral emotions.
        - Redistributes unused quotas from underrepresented classes.

    Args:
        df: MELD dataframe containing 'audio_path' and 'emotion'.
        target_total: Desired number of samples.
        max_neutral_pct: Maximum proportion of neutral samples.
        min_duration: Minimum audio duration in seconds.
        max_duration: Maximum audio duration in seconds.
        seed: Random seed for deterministic sampling.

    Returns:
        Balanced and filtered DataFrame.
    """

    df = df.copy()

    # ----------------------------------------------------------
    # 1. Deterministic ordering
    # ----------------------------------------------------------

    id_cols = [
        c for c in ["dialogue", "utterance"]
        if c in df.columns
    ]

    if id_cols:
        df = df.sort_values(id_cols)
    elif "audio_path" in df.columns:
        df = df.sort_values("audio_path")

    df = df.reset_index(drop=True)

    # ----------------------------------------------------------
    # 2. Validate audio paths
    # ----------------------------------------------------------

    if "audio_path" not in df.columns:
        raise ValueError("DataFrame must contain 'audio_path'.")

    df = df[
        df["audio_path"].notna()
        & df["audio_path"].apply(
            lambda p: Path(str(p)).is_file()
        )
    ].copy()

    print(f"After audio validation: {len(df):,}")

    # ----------------------------------------------------------
    # 3. Audio duration filtering
    # ----------------------------------------------------------

    print("Computing audio durations...")

    df["duration"] = df["audio_path"].apply(get_audio_duration)

    failed_duration = df["duration"].isna().sum()

    if failed_duration:
        print(
            f"Warning: could not determine duration for "
            f"{failed_duration:,} files."
        )

    df = df[df["duration"].notna()].copy()

    df = df[
        df["duration"].between(
            min_duration,
            max_duration,
            inclusive="both",
        )
    ].copy()

    print(
        f"Duration filter: "
        f"{min_duration:.1f}s–{max_duration:.1f}s"
    )
    print(f"After duration filtering: {len(df):,}")

    # ----------------------------------------------------------
    # 4. Normalize emotion labels
    # ----------------------------------------------------------

    df["emotion"] = (
        df["emotion"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df = df[df["emotion"].ne("nan")].copy()

    emotions = sorted(df["emotion"].unique())

    if not emotions:
        raise ValueError("No valid emotion classes remain.")

    print("\nAvailable emotions:")
    print(emotions)

    # ----------------------------------------------------------
    # 5. Cap neutral
    # ----------------------------------------------------------

    max_neutral_count = int(
        target_total * max_neutral_pct
    )

    non_neutral = [
        emotion
        for emotion in emotions
        if emotion != "neutral"
    ]

    # If neutral is not present, all samples go to
    # non-neutral classes.
    neutral_target = (
        min(
            max_neutral_count,
            len(df[df["emotion"] == "neutral"])
        )
        if "neutral" in emotions
        else 0
    )

    remaining_slots = target_total - neutral_target

    # ----------------------------------------------------------
    # 6. Initial non-neutral quotas
    # ----------------------------------------------------------

    quotas = {}

    if non_neutral:
        base = remaining_slots // len(non_neutral)
        remainder = remaining_slots % len(non_neutral)

        for i, emotion in enumerate(non_neutral):
            quotas[emotion] = (
                base + (1 if i < remainder else 0)
            )

    if "neutral" in emotions:
        quotas["neutral"] = neutral_target

    # ----------------------------------------------------------
    # 7. Redistribute unavailable samples
    # ----------------------------------------------------------

    while True:
        deficit = 0
        saturated = set()

        for emotion, quota in quotas.items():
            available = (
                df["emotion"].eq(emotion).sum()
            )

            if available < quota:
                deficit += quota - available
                quotas[emotion] = available
                saturated.add(emotion)

        if deficit == 0:
            break

        eligible = [
            emotion
            for emotion in non_neutral
            if emotion not in saturated
            and quotas[emotion]
            < df["emotion"].eq(emotion).sum()
        ]

        if not eligible:
            break

        base_extra = deficit // len(eligible)
        remainder = deficit % len(eligible)

        for i, emotion in enumerate(eligible):
            quotas[emotion] += (
                base_extra
                + (1 if i < remainder else 0)
            )

    # ----------------------------------------------------------
    # 8. Deterministic sampling
    # ----------------------------------------------------------

    sampled = []

    for emotion in sorted(quotas):
        n = quotas[emotion]

        if n == 0:
            continue

        class_df = df[df["emotion"] == emotion]

        sampled.append(
            class_df.sample(
                n=n,
                random_state=seed,
                replace=False,
            )
        )

    if not sampled:
        raise ValueError(
            "No samples available after filtering."
        )

    result = pd.concat(
        sampled,
        ignore_index=True,
    )

    # ----------------------------------------------------------
    # 9. Final deterministic ordering
    # ----------------------------------------------------------

    if id_cols:
        result = result.sort_values(id_cols)

    result = result.reset_index(drop=True)

    # ----------------------------------------------------------
    # 10. Diagnostics
    # ----------------------------------------------------------

    print("\n" + "=" * 50)
    print("MELD SAMPLING SUMMARY")
    print("=" * 50)

    print(f"Requested samples: {target_total:,}")
    print(f"Returned samples:  {len(result):,}")

    print("\nEmotion distribution:")
    print(result["emotion"].value_counts())

    print("\nEmotion proportions:")
    print(
        result["emotion"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .astype(str)
        .add("%")
    )

    print("\nAudio duration:")
    print(
        result["duration"].describe()[
            ["min", "mean", "max"]
        ].round(2)
    )

    print("\nQuotas:")
    print(quotas)

    return result

In [ ]:
pruned_train_set = sample_balanced_meld(
    raw_train_set,
    target_total=2000,
    max_neutral_pct=0.15,
    min_duration=0.5,
    max_duration=30.0,
    seed=42,
)

run_test = sample_balanced_meld(
    raw_train_set,
    target_total=2000,
    max_neutral_pct=0.15,
    min_duration=0.5,
    max_duration=30.0,
    seed=42,
)

assert pruned_train_set.equals(run_test)

print("\nFinal candidate pool:")
display(pruned_train_set.head())

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_set, val_set = train_test_split(
    pruned_train_set,
    test_size=0.2,
    random_state=42,
    stratify=pruned_train_set["emotion"],
)

In [ ]:
train_set['emotion'].value_counts()

In [ ]:
train_set['speaker'].value_counts()

In [ ]:
display(train_set)

# Model Config
Qwen 2.5 Omni - 4bit Quantization

In [ ]:
import torch
from typing import Dict, Any, Tuple
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

## Prompt

In [ ]:
qwen_persona = (
    "You are an expert in speech emotion recognition and acoustic phonetics. "
    "You evaluate vocal affect strictly through physical acoustic properties."
)

cot_prompt = """Listen to the audio clip and predict the speaker's emotional state solely from acoustics.

Allowed emotions: anger, joy, sadness, surprise, fear, disgust, neutral

Instructions:
1. Complete [Acoustic Inventory] selecting one option per line.
2. In [Prosodic Reasoning], deduce which emotion best matches these physical cues and eliminate conflicting states.
3. Output only the predicted class inside <answer>.

Output strictly in this format:
<think>
[Acoustic Inventory]
- Pitch Height: low | moderate | elevated
- Pitch Dynamics: narrow | moderate | wide
- Vocal Energy: quiet | moderate | loud
- Speaking Rate: slow | moderate | fast
- Voice Quality: pressed/tense | breathy | harsh/creaky | modal/normal

[Prosodic Reasoning]
(Deduce the emotion from the cues above)
</think>
<answer>emotion</answer>"""

## Quantization config

In [ ]:
!pip install -q trl
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.6 MB/s eta 0:00:00


In [ ]:
from transformers import (
    AutoProcessor,
    Qwen2_5OmniForConditionalGeneration,
    BitsAndBytesConfig
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

## LoRA Config

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.00,
    bias="none",
    task_type=None,
    target_modules=r"thinker\.model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
)

## General Config

In [ ]:
model_id = "Qwen/Qwen2.5-Omni-7B"

# Reward Functions

## Utils

In [ ]:
VALID_EMOTIONS =  {"anger", "joy", "sadness", "surprise", "fear", "disgust", "neutral"}

In [ ]:
LABEL_SYNONYMS = {
    "happy": "joy",
    "happiness": "joy",
    "sad": "sadness",
    "mad": "anger",
    "angry": "anger",
    "scared": "fear",
    "fearful": "fear",
    "disgusted": "disgust",
    "surprised": "surprise",
}

SET_VALID_EMOTIONS = set(VALID_EMOTIONS)

In [ ]:
REQUIRED_INVENTORY_KEYS = [
    "pitch_height",
    "pitch_dynamics",
    "vocal_energy",
    "speaking_rate",
    "voice_quality",
]

In [ ]:
import json
from pathlib import Path
import re
from typing import Any, Dict, List, Optional
from collections import Counter

In [ ]:
def extract_acoustic_inventory(completion: str) -> Dict[str, Optional[str]]:
    section_match = re.search(
        r"\[Acoustic Inventory\](.*?)(?:\[Prosodic Reasoning\]|</think>|$)",
        completion,
        re.DOTALL | re.IGNORECASE,
    )
    target_text = section_match.group(1) if section_match else completion

    patterns = {
        "pitch_height": (
            r"Pitch Height\s*:\s*(low|moderate|elevated)\b",
            {"low": "low", "moderate": "moderate", "elevated": "elevated"},
        ),
        "pitch_dynamics": (
            r"Pitch Dynamics\s*:\s*(narrow|moderate|wide)\b",
            {"narrow": "narrow", "moderate": "moderate", "wide": "wide"},
        ),
        "vocal_energy": (
            r"Vocal Energy\s*:\s*(quiet|moderate|loud)\b",
            {"quiet": "quiet", "moderate": "moderate", "loud": "loud"},
        ),
        "speaking_rate": (
            r"Speaking Rate\s*:\s*(slow|moderate|fast)\b",
            {"slow": "slow", "moderate": "moderate", "fast": "fast"},
        ),
        "voice_quality": (
            r"Voice Quality\s*:\s*(pressed/tense|pressed|tense|breathy|harsh/creaky|harsh|creaky|modal/normal|modal|normal)\b",
            {
                "pressed/tense": "pressed/tense",
                "pressed": "pressed/tense",
                "tense": "pressed/tense",
                "breathy": "breathy",
                "harsh/creaky": "harsh/creaky",
                "harsh": "harsh/creaky",
                "creaky": "harsh/creaky",
                "modal/normal": "modal/normal",
                "modal": "modal/normal",
                "normal": "modal/normal",
            },
        ),
    }

    parsed: Dict[str, Optional[str]] = {}

    for key, (pattern, canonical_map) in patterns.items():
        match = re.search(pattern, target_text, re.IGNORECASE)
        if match:
            raw_val = match.group(1).lower().strip()
            parsed[key] = canonical_map.get(raw_val, raw_val)
        else:
            parsed[key] = None

    return parsed

In [ ]:
def extract_reasoning(completion: str) -> str:
    match = re.search(
        r"\[Prosodic Reasoning\](.*?)(?:</think>|<answer>|$)",
        completion,
        re.DOTALL | re.IGNORECASE,
    )
    if not match:
        return ""

    reasoning = match.group(1).strip()

    placeholder = "(Deduce the emotion from the cues above)"
    if reasoning.startswith(placeholder):
        reasoning = reasoning[len(placeholder) :].strip()

    return reasoning

In [ ]:
def extract_answer_content(completion: str) -> str:
    match = re.search(
        r"<answer>(.*?)(?:</answer>|$)", completion, re.DOTALL | re.IGNORECASE
    )
    if not match:
        return ""

    raw_text = match.group(1).strip().lower()

    cleaned = re.sub(r"[\[\]\"\'\*\.]", "", raw_text).strip()

    if cleaned in VALID_EMOTIONS:
        return cleaned

    for emotion in VALID_EMOTIONS:
        if re.search(rf"\b{re.escape(emotion)}\b", cleaned):
            return emotion

    return cleaned

In [ ]:
def compute_emotion_weights(
    train_set: Any, power: float = 0.4
) -> Dict[str, float]:
    """Computes smoothed inverse class-frequency weights for GRPO rewards.

    Uses w_c = (max_count / count) ** power so the majority class (neutral)
    is anchored at 1.0, and minority classes receive an advantage boost
    without blowing up gradient variance.

    Args:
        train_set: Pandas DataFrame, HuggingFace Dataset, or list containing an
          'emotion' column/key.
        power: Smoothing exponent. 0.0 = uniform (all 1.0), 0.4-0.5 = smoothed
          inverse (recommended for RLVR), 1.0 = raw inverse frequency.
    """
    emotions = list(train_set["emotion"])
    counts = Counter(emotions)
    max_count = max(counts.values())

    weights = {
        emotion: round((max_count / count) ** power, 3)
        for emotion, count in counts.items()
    }
    return weights

CLASS_WEIGHTS = compute_emotion_weights(train_set)

In [ ]:
display(CLASS_WEIGHTS)

{'neutral': 1.0,
 'surprise': 1.725,
 'fear': 3.147,
 'sadness': 2.165,
 'joy': 1.488,
 'disgust': 3.133,
 'anger': 1.783}

## R1 - Accuracy

In [ ]:
def reward_label_accuracy_weighted(
    prompts: List[str],
    completions: List[str],
    label: List[str],
    **kwargs
) -> List[float]:
    rewards = []
    for comp, target in zip(completions, label):
        answer = extract_answer_content(comp)
        pred_norm = LABEL_SYNONYMS.get(answer, answer)

        target_norm = str(target).lower().strip()
        target_norm = LABEL_SYNONYMS.get(target_norm, target_norm)

        # label rarity sensible reward
        if pred_norm == target_norm:
            weight = CLASS_WEIGHTS.get(target_norm, 1.0)
            rewards.append(float(weight))
        else:
            rewards.append(0.0)

    return rewards

## R2 - Acoustic Inventory

## R3 - Format

In [ ]:
def compute_format_reward(
    completion: str,
    parsed_inventory: Dict[str, Optional[str]],
    extracted_reasoning: str,
    extracted_answer: str,
) -> float:
    """Computes a graded format reward in [0.0, 1.0] to guide early GRPO rollouts

    without incentivizing reward hacking.
    """
    reward = 0.0
    has_think_open = "<think>" in completion
    has_think_close = "</think>" in completion
    has_answer_open = "<answer>" in completion
    has_answer_close = "</answer>" in completion

    if (
        has_think_open
        and has_think_close
        and has_answer_open
        and has_answer_close
    ):
        t_open = completion.find("<think>")
        t_close = completion.find("</think>")
        a_open = completion.find("<answer>")
        a_close = completion.find("</answer>")
        if t_open < t_close < a_open < a_close:
            reward += 0.20

    valid_slots = sum(
        1
        for key in REQUIRED_INVENTORY_KEYS
        if parsed_inventory.get(key) is not None
    )
    reward += 0.35 * (valid_slots / 5.0)

    clean_reasoning = extracted_reasoning.strip()
    placeholder = "(Deduce the emotion from the cues above)"

    if len(clean_reasoning) >= 20 and clean_reasoning != placeholder:
        words = clean_reasoning.split()
        if 5 <= len(words) <= 150:
            reward += 0.25
        elif len(words) > 150:
            reward += 0.10

    if extracted_answer in VALID_EMOTIONS:
        reward += 0.20

    return round(reward, 3)

# Model and Processor Load

In [ ]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.disable_talker()
model.generation_config.return_audio = False
if hasattr(model, "talker_config"):
    model.config.enable_talker = False

if hasattr(processor, "generation_config") and processor.generation_config is not None:
    processor.generation_config.return_audio = False
model = prepare_model_for_kbit_training(model)#, use_gradient_checkpointing=True)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()